# 🔬 Extra — scales, twin axes and seaborn

**Extra material.** Nothing in the assessed work assumes this notebook.

**Time:** ~25 minutes &nbsp;•&nbsp; **Goal:** three things that come up constantly once you start plotting your own data — and one library that does a lot of this for you, which is only safe to use once you know what it is doing.

## Run this first

In [ ]:
# Run this first. Every exercise below uses `df`.
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab07/data/"

DEPMAP = "depmap.csv"
if not Path(DEPMAP).exists():
    urllib.request.urlretrieve(DATA_URL + DEPMAP, DEPMAP)

df = pd.read_csv(DEPMAP)

GENES = ["SOX10", "BRAF", "KRAS", "NRAS", "PAX8", "MYB", "CTNNB1", "EGFR",
         "MDM2", "ESR1", "RAN", "RPL5", "RPS6", "PSMA1", "SNRPD1", "EIF4A3",
         "A1BG", "AADAC"]

print(df.shape)
df[["cell_line_name", "oncotree_lineage", "SOX10", "BRAF", "RAN"]].head()

## Skill 1 — Log scales, and when not to use one

A log axis compresses a long right tail so you can see the small values. It is the right move for counts, concentrations, expression values, gene lengths — anything spanning orders of magnitude.

```python
ax.set_yscale("log")
ax.set_xscale("log")
```

It is the **wrong** move for a dependency score, for two reasons: the values are negative, and the scale is already additive rather than multiplicative. A score of −4 is not "ten times" −0.4, it is a different amount of the same quantity.

So to demonstrate a log scale honestly we need something that actually is long-tailed. The *counts* in a histogram are.

In [ ]:
# DEMO - a log COUNT axis, which is legitimate
fig, axes = plt.subplots(1, 2)
fig.set_size_inches(11, 3.5)

for ax, scale in zip(axes, ["linear", "log"]):
    ax.hist(df["SOX10"], bins=40, color="#5b9df9")
    ax.set_yscale(scale)
    ax.set_xlabel("SOX10 dependency")
    ax.set_ylabel(f"cell lines ({scale} scale)")

fig.tight_layout()
plt.show()

On the left, the tail is a few pixels high and you have to take its existence on trust. On the right, every bin with even one cell line in it is visible, and you can see the tail runs all the way to −3.

That is what a log count axis is for: **rare things become visible without the common things being hidden.** Say in the caption that the axis is logged, every time — a reader who misses it will misjudge every height on the plot.

### 🔬 DIY challenge

1. Do the same for `RAN`. Does the log axis reveal anything, and if not, why not?
2. `ax.set_yscale("symlog")` handles data spanning zero. Try it on a scatter of `SOX10` against `BRAF`.
3. Write one sentence explaining why `ax.set_xscale("log")` on a dependency score would be meaningless.

> 💡 The test for a log axis: does the quantity have a true zero, and do you care about **ratios** between values? Counts yes; dependency scores no.

In [ ]:
# TODO - your turn

## Skill 2 — Two y-axes with `twinx`

`ax.twinx()` gives you a second axes sharing the same x-axis but with its own y-axis on the right. It is the standard way to put two quantities with different units on one plot.

```python
ax2 = ax.twinx()
ax2.plot(x, other_thing, color="#d6a44c")
ax2.set_ylabel("something else")
```

It is also the most commonly abused chart in science. Two y-axes can be scaled to make any two series appear correlated, and readers know that — so use it when the x-axis genuinely is shared and the two quantities genuinely are different units, and colour-code each axis to its series so nobody has to guess which is which.

In [ ]:
# DEMO - how many lines per lineage, and how dependent they are on SOX10
counts = df["oncotree_lineage"].value_counts().head(10)
means = df.groupby("oncotree_lineage")["SOX10"].mean()[counts.index]

fig, ax = plt.subplots()
fig.set_size_inches(8, 3.8)

BAR, LINE = "#9ca3af", "#5b9df9"

ax.bar(counts.index, counts.values, color=BAR)
ax.set_ylabel("cell lines", color=BAR)
ax.tick_params(axis="y", labelcolor=BAR)
ax.tick_params(axis="x", rotation=45, labelsize=9)

ax2 = ax.twinx()
ax2.plot(counts.index, means.values, color=LINE, marker="o", linewidth=2)
ax2.axhline(0, color="#d1d5db", linewidth=0.8)
ax2.set_ylabel("mean SOX10 dependency", color=LINE)
ax2.tick_params(axis="y", labelcolor=LINE)

ax.set_title("Panel size and SOX10 dependency are unrelated — as they should be")
fig.tight_layout()
plt.show()

Note what the title says. The honest reading of this chart is that the two series have **nothing to do with each other** — the blue line is flat everywhere except skin, and the grey bars vary independently. That is a useful thing to show, and it is the only kind of claim a twin-axis plot supports well: *these move together* or *these do not*.

### 🔬 DIY challenge

1. Rescale `ax2` with `ax2.set_ylim(-1.5, 1.5)` and then with `ax2.set_ylim(-1.3, -1.2)`. Watch how much the apparent relationship changes. This is why people distrust the chart.
2. Redraw it as two stacked panels sharing an x-axis instead. Which version would you put in a report?
3. `ax.twiny()` is the horizontal equivalent. When might that be useful?

> 💡 Question 2 usually wins. A twin axis saves space; two panels save an argument with a reviewer.

In [ ]:
# TODO - your turn

## Skill 3 — seaborn, and what it does for you

`seaborn` is a layer on top of matplotlib. It takes a DataFrame, a couple of column names, and does the grouping, the colouring and the legend for you.

```python
import seaborn as sns
sns.boxplot(data=df, x="oncotree_lineage", y="SOX10", ax=ax)
```

Compare that to notebook 1, where you wrote a list comprehension to build one array per group. seaborn does that step internally.

Two things are worth being clear about:

- **it returns a matplotlib `ax`.** Every method you learned still applies — `ax.set_xlabel`, `ax.axhline`, spines, `fig.savefig`. seaborn is not a different plotting system.
- **it makes decisions for you**, about colour, about error bars, about which summary to show. Those decisions are usually good, but they are decisions, and you should be able to say what they were.

That is why this notebook comes last rather than first.

In [ ]:
# DEMO - the boxplot from notebook 1, the seaborn way
import seaborn as sns

names = ["Skin", "Lung", "Lymphoid", "Breast", "CNS/Brain"]
subset = df[df["oncotree_lineage"].isin(names)]

fig, ax = plt.subplots()
fig.set_size_inches(7, 3.5)

sns.boxplot(data=subset, x="oncotree_lineage", y="SOX10",
            order=names, color="#9ca3af", ax=ax)

# still an ordinary matplotlib axes
ax.axhline(0, color="#374151", linewidth=0.8)
ax.set_xlabel("")
ax.set_ylabel("SOX10 dependency")
ax.set_title("Same figure, no list comprehension")
plt.show()

In [ ]:
# DEMO - and the thing seaborn is genuinely better at
fig, ax = plt.subplots()
fig.set_size_inches(7, 3.5)

sns.stripplot(data=subset, x="oncotree_lineage", y="SOX10",
              order=names, hue="oncotree_lineage", legend=False,
              size=3, alpha=0.6, ax=ax)
sns.boxplot(data=subset, x="oncotree_lineage", y="SOX10",
            order=names, showfliers=False, boxprops=dict(alpha=0.3),
            color="#9ca3af", ax=ax)

ax.axhline(0, color="#374151", linewidth=0.8)
ax.set_xlabel("")
ax.set_ylabel("SOX10 dependency")
ax.set_title("Every cell line, and the summary on top")
plt.show()

That second figure is the one to remember. A boxplot summarises; a stripplot shows you **every data point**. Overlaying them means the reader can see both the summary and the thing it summarises — and can tell when a box is built on eight points.

For small-to-medium n, this is very close to the best general-purpose comparison plot there is, and drawing it in raw matplotlib is a genuine nuisance.

### 🏆 DIY finale

1. Rebuild your best figure from notebook 3 using seaborn where it helps, and say which panels got shorter.
2. `sns.violinplot` shows the distribution shape rather than the quartiles. Try it on the same data — does it help or overstate?
3. `sns.pairplot(df[["SOX10", "BRAF", "NRAS", "MDM2"]])` draws every pairwise scatter at once. Run it, then say what you would use it for and what you would never put in a paper.
4. `sns.set_theme()` sets a whole style. Run it, redraw a matplotlib-only figure from notebook 1, and confirm the change applies there too — which tells you exactly what seaborn is.

> 💡 Question 4 is the point of this notebook. seaborn sets `rcParams`. It has been matplotlib all along.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- The seaborn tutorial, which is unusually good: <https://seaborn.pydata.org/tutorial.html>
- Scales in matplotlib, including `symlog` and `logit`: <https://matplotlib.org/stable/gallery/scales/scales.html>
- Claus Wilke, *Fundamentals of Data Visualization* — free online, and the best book on this: <https://clauswilke.com/dataviz/>

## Recap

You can now:

- use a log axis where ratios are meaningful, and say why a dependency score is not such a case
- build a twin-axis plot, and know why readers distrust one
- use seaborn for the grouping-and-colouring boilerplate, while keeping the `ax` you know how to control
- overlay a stripplot on a boxplot, which is often the most honest comparison plot available